# Gesture CNN-LSTM

CSV из подпапок `ml/dataset/data/<gesture>/` становятся жестами. CSV прямо в `data/` режутся на шумовые окна `__noise__`. Временная сетка общая для обучения и live-распознавания: `SEQ_LEN` точек с шагом `GRID_DT`, forward-fill между событиями.


In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torch.utils.data import DataLoader


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "ml" / "gesture_lstm" / "dataset.py").is_file():
            return p
        p = p.parent
    raise RuntimeError("Откройте ноутбук из репозитория KinestheticBook.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml.gesture_lstm.dataset import GestureNoiseDataset, NormalizedSubset, SubsetDataset, compute_norm_stats, stratified_split
from ml.gesture_lstm.model import GestureCNNLSTM, GestureLSTM

print(REPO_ROOT)
print(torch.__version__, "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
DATA_DIR = REPO_ROOT / "ml" / "dataset" / "data"
OUT_DIR = REPO_ROOT / "ml" / "gesture_lstm" / "runs" / "notebook"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_TYPE = "cnn_lstm"  # "cnn_lstm" или "lstm"
SEQ_LEN = 64
GRID_DT = 0.05
NOISE_WINDOW = 48
NOISE_STRIDE = 24
MAX_NOISE_SAMPLES = 400

HIDDEN = 64
CNN_CHANNELS = 64
LSTM_LAYERS = 2
DROPOUT = 0.25
EPOCHS = 80
BATCH_SIZE = 16
LR = 1e-3
VAL_RATIO = 0.2
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
full_ds = GestureNoiseDataset(
    DATA_DIR,
    seq_len=SEQ_LEN,
    grid_dt=GRID_DT,
    noise_window_rows=NOISE_WINDOW,
    noise_stride=NOISE_STRIDE,
    max_noise_samples=MAX_NOISE_SAMPLES,
    seed=SEED,
)

counts = Counter(y for _, y, _ in full_ds.samples)
print("samples:", len(full_ds))
print("labels:", full_ds.label_to_idx)
for i, n in sorted(counts.items()):
    print(f"{full_ds.idx_to_label[i]}: {n}")


In [ ]:
train_idx, val_idx = stratified_split(full_ds, val_ratio=VAL_RATIO, seed=SEED)
mean, std = compute_norm_stats(full_ds, train_idx)
np.savez(OUT_DIR / "norm.npz", mean=mean, std=std)

train_ds = NormalizedSubset(SubsetDataset(full_ds, train_idx, augment=True), mean, std)
val_ds = NormalizedSubset(SubsetDataset(full_ds, val_idx, augment=False), mean, std)
loader_train = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

n_classes = full_ds.num_classes()
if MODEL_TYPE == "cnn_lstm":
    model = GestureCNNLSTM(len(mean), CNN_CHANNELS, HIDDEN, LSTM_LAYERS, n_classes, DROPOUT)
else:
    model = GestureLSTM(len(mean), HIDDEN, LSTM_LAYERS, n_classes, DROPOUT)
model = model.to(DEVICE)

opt = torch.optim.Adam(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()
print("train:", len(train_idx), "val:", len(val_idx), "classes:", n_classes)


In [ ]:
best_val = float("inf")
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = correct = 0
    loss_sum = 0.0
    for x, y in loader_train:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        logits = model(x)
        loss = crit(logits, y)
        loss.backward()
        opt.step()

        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)

    model.eval()
    v_total = v_correct = 0
    v_loss = 0.0
    with torch.no_grad():
        for x, y in loader_val:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = crit(logits, y)
            v_loss += loss.item() * x.size(0)
            v_correct += (logits.argmax(1) == y).sum().item()
            v_total += x.size(0)

    train_loss = loss_sum / max(total, 1)
    train_acc = correct / max(total, 1)
    val_loss = v_loss / max(v_total, 1)
    val_acc = v_correct / max(v_total, 1)

    if epoch == 1 or epoch % max(1, EPOCHS // 10) == 0 or epoch == EPOCHS:
        print(f"{epoch:03d}: train {train_loss:.4f}/{train_acc:.3f}  val {val_loss:.4f}/{val_acc:.3f}")

    if v_total and val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state or model.state_dict())
print("best val_loss:", best_val)


In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in loader_val:
        pred = model(x.to(DEVICE)).argmax(1).cpu().numpy().tolist()
        y_pred.extend(pred)
        y_true.extend(y.numpy().tolist())

labels = list(range(n_classes))
names = [full_ds.idx_to_label[i] for i in labels]

if y_true:
    print(classification_report(y_true, y_pred, labels=labels, target_names=names, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(max(6, n_classes * 1.3), max(5, n_classes)))
    ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Validation split is empty.")


In [ ]:
ckpt = {
    "model_state": model.state_dict(),
    "config": {
        "model_type": MODEL_TYPE,
        "n_features": int(len(mean)),
        "hidden_size": HIDDEN,
        "cnn_channels": CNN_CHANNELS if MODEL_TYPE == "cnn_lstm" else None,
        "num_layers": LSTM_LAYERS,
        "num_classes": n_classes,
        "dropout": DROPOUT,
        "seq_len": SEQ_LEN,
        "grid_dt": GRID_DT,
    },
    "label_to_idx": full_ds.label_to_idx,
}
torch.save(ckpt, OUT_DIR / "checkpoint.pt")
full_ds.export_label_map(OUT_DIR / "label_map.json")

meta = {
    "data_dir": str(DATA_DIR.resolve()),
    "n_samples": len(full_ds),
    "n_train": len(train_idx),
    "n_val": len(val_idx),
    "classes": full_ds.label_to_idx,
    "model_type": MODEL_TYPE,
    "grid_dt": GRID_DT,
}
(OUT_DIR / "run_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved to", OUT_DIR)
